# 基于 PyTorch 的多层感知机（MLP）识别 MNIST

**实验目标：**
1. 学习使用 PyTorch 进行深度学习模型的设计和训练。
2. 实现一个多层感知机（MLP）模型，掌握数据加载、模型构建、训练、评估的基本流程。
3. 了解深度学习模型的调优方法，包括优化器、损失函数的选择等。

In [ ]:
# 导入必要的库
import torch
from torch import nn
import torch.optim as optim
import torch.utils.data as Data
import torchvision
import torchvision.transforms as transforms

# 设置随机种子以确保结果可重现
torch.manual_seed(1)

# 选择训练设备：有 GPU 就用 GPU，没有就用 CPU
device: torch.device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("PyTorch 版本:", torch.__version__)
print("使用设备:", device)

## 1. 了解并加载 MNIST 数据集

MNIST 是手写数字识别数据集，包含 0-9 共 10 个类别：
- 训练集：60000 张 28×28 的灰度图像；
- 测试集：10000 张 28×28 的灰度图像；
- 每张图像对应一个标签（0-9）。

我们使用 `torchvision.datasets.MNIST` 直接下载并加载数据。

In [ ]:
# 定义数据预处理：将 PIL 图像转换为 Tensor，并归一化到 [-1, 1]
# MNIST 训练集的均值约为 0.1307，标准差约为 0.3081
transform = transforms.Compose(
    [
        transforms.ToTensor(),  # PIL Image -> Tensor, 像素从 [0,255] 缩放到 [0,1]
        transforms.Normalize((0.1307,), (0.3081,)),  # 按均值/标准差做归一化
    ]
)

# 下载并加载训练集
mnist_train = torchvision.datasets.MNIST(
    root="../data",  # 数据存放目录
    train=True,  # 训练集
    download=True,  # 若本地没有则自动下载
    transform=transform,  # 应用预处理
)

# 下载并加载测试集
mnist_test = torchvision.datasets.MNIST(
    root="../data",
    train=False,  # 测试集
    download=True,
    transform=transform,
)

print(f"训练集样本数: {len(mnist_train)}")
print(f"测试集样本数: {len(mnist_test)}")

# 查看单张图像的形状和标签
image, label = mnist_train[0]
print(f"图像张量形状: {image.shape}  (通道, 高, 宽)")
print(f"标签: {label}")

In [ ]:
# 可视化几张训练集图像，直观感受数据
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img, lbl = mnist_train[i]
    # 反归一化以便显示（仅用于展示）
    ax.imshow(img.squeeze().numpy() * 0.3081 + 0.1307, cmap="gray")
    ax.set_title(f"label: {lbl}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. 构建 DataLoader

`DataLoader` 负责将数据集按 batch 切分，并在训练时进行打乱。

In [ ]:
# 批量大小：较大的 batch 训练更稳定，但占用更多内存
batch_size: int = 128

train_iter = Data.DataLoader(
    dataset=mnist_train,
    batch_size=batch_size,
    shuffle=True,  # 训练集需要打乱
    num_workers=0,
)

test_iter = Data.DataLoader(
    dataset=mnist_test,
    batch_size=batch_size,
    shuffle=False,  # 测试集不需要打乱
    num_workers=0,
)

# 查看一个 batch 的结构
X: torch.Tensor
y: torch.Tensor
for X, y in train_iter:
    print(f"一个 batch 的图像形状: {X.shape}")  # [batch, 1, 28, 28]
    print(f"一个 batch 的标签形状: {y.shape}")  # [batch]
    break

## 3. 定义多层感知机（MLP）模型

MLP 结构：
- 输入层：784 维（将 28×28 的图像展平）
- 隐藏层 1：256 个神经元，ReLU 激活
- 隐藏层 2：128 个神经元，ReLU 激活
- 输出层：10 维（对应 10 个数字类别）

> 注意：我们不在最后一层加 Softmax，因为后面使用的 `nn.CrossEntropyLoss`
> 内部已经包含了 `LogSoftmax`，直接接收原始 logits 更稳定。

In [ ]:
# 定义网络结构常量
num_inputs: int = 28 * 28  # 输入维度：784
num_hidden1: int = 256  # 第一隐藏层神经元数
num_hidden2: int = 128  # 第二隐藏层神经元数
num_outputs: int = 10  # 输出类别数


# 方法 1：通过继承 nn.Module 自定义 MLP
class MLP(nn.Module):
    def __init__(self) -> None:
        super(MLP, self).__init__()
        # nn.Flatten 将 [batch, 1, 28, 28] 展平为 [batch, 784]
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(num_inputs, num_hidden1)
        self.hidden2 = nn.Linear(num_hidden1, num_hidden2)
        self.output = nn.Linear(num_hidden2, num_outputs)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        # 输出未经 softmax 的 logits
        return self.output(x)


# 方法 2：使用 nn.Sequential 构建相同结构（更简洁）
net: nn.Sequential = nn.Sequential(
    nn.Flatten(),
    nn.Linear(num_inputs, num_hidden1),
    nn.ReLU(),
    nn.Linear(num_hidden1, num_hidden2),
    nn.ReLU(),
    nn.Linear(num_hidden2, num_outputs),
)

# 将模型搬到计算设备
net = net.to(device)
print(net)

## 4. 初始化模型参数

使用 Xavier（Glorot）正态初始化权重，偏置初始化为 0。
良好的初始化能让训练更稳定、收敛更快。

In [ ]:
from torch.nn import init


def init_weights(m: nn.Module) -> None:
    if isinstance(m, nn.Linear):
        init.xavier_normal_(m.weight)
        init.constant_(m.bias, val=0.0)


# apply 会递归地将该函数应用到每个子模块上
net.apply(init_weights)

# 打印一部分参数，确认初始化已生效
for name, param in net.named_parameters():
    print(name, param.shape)

## 5. 定义损失函数

多分类任务常用 **交叉熵损失 `nn.CrossEntropyLoss`**：
- 它内部会先对 logits 做 `LogSoftmax`，再计算 `NLLLoss`；
- 标签应为 `LongTensor`，取值为 `[0, num_classes-1]`。

In [ ]:
loss: nn.CrossEntropyLoss = nn.CrossEntropyLoss()

## 6. 定义优化器

这里使用 Adam 优化器，它对学习率不太敏感，通常比原始 SGD 收敛更快。
也可以尝试替换为 `optim.SGD(net.parameters(), lr=0.1, momentum=0.9)` 做对比。

In [ ]:
lr: float = 1e-3
optimizer: optim.Adam = optim.Adam(net.parameters(), lr=lr)
print(optimizer)

## 7. 定义评估函数

在测试集上统计准确率（accuracy）。在评估阶段需要：
1. `net.eval()` 关闭 Dropout / BatchNorm 的训练模式；
2. 用 `torch.no_grad()` 关闭梯度记录，节省显存、加速推理。

In [ ]:
def evaluate_accuracy(data_iter: Data.DataLoader, net: nn.Module) -> float:
    net.eval()
    correct: int = 0
    total: int = 0
    with torch.no_grad():
        for X, y in data_iter:
            X = X.to(device)
            y = y.to(device)
            # argmax(dim=1) 取每个样本得分最高的类别作为预测
            y_hat: torch.Tensor = net(X).argmax(dim=1)
            correct += (y_hat == y).sum().item()
            total += y.numel()
    net.train()
    return correct / total

## 8. 训练模型

训练循环的标准五步：
1. 前向传播 → 2. 计算损失 → 3. 梯度清零 → 4. 反向传播 → 5. 更新参数。

In [ ]:
num_epochs: int = 10

for epoch in range(1, num_epochs + 1):
    net.train()
    train_loss_sum: float = 0.0  # 累计损失
    train_acc_sum: int = 0  # 累计预测正确的样本数
    n: int = 0  # 累计样本数

    for X, y in train_iter:
        X = X.to(device)
        y = y.to(device)

        # 前向传播
        y_hat: torch.Tensor = net(X)
        # 计算损失
        batch_loss: torch.Tensor = loss(y_hat, y)

        # 梯度清零
        optimizer.zero_grad()
        # 反向传播
        batch_loss.backward()
        # 更新参数
        optimizer.step()

        # 统计：损失乘以 batch 大小便于后续求平均
        train_loss_sum += batch_loss.item() * y.shape[0]
        train_acc_sum += (y_hat.argmax(dim=1) == y).sum().item()
        n += y.shape[0]

    test_acc: float = evaluate_accuracy(test_iter, net)
    print(
        f"epoch {epoch:2d}, "
        f"loss {train_loss_sum / n:.4f}, "
        f"train acc {train_acc_sum / n:.4f}, "
        f"test acc {test_acc:.4f}"
    )

## 9. 查看预测结果

从测试集中取一个 batch，可视化模型的预测结果。

In [ ]:
net.eval()
X, y = next(iter(test_iter))
X_dev = X.to(device)
with torch.no_grad():
    preds: torch.Tensor = net(X_dev).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i].squeeze().numpy() * 0.3081 + 0.1307, cmap="gray")
    color = "green" if preds[i].item() == y[i].item() else "red"
    ax.set_title(
        f"pred: {preds[i].item()}\nlabel: {y[i].item()}", color=color, fontsize=9
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. 调优思考（课后练习）

可以尝试以下方向改进模型并对比结果：
1. **网络结构**：增减隐藏层数量 / 神经元数量。
2. **激活函数**：将 ReLU 换成 Sigmoid、Tanh、LeakyReLU 等。
3. **优化器**：对比 SGD、SGD+Momentum、Adam、AdamW。
4. **学习率**：调整 lr 或使用 `torch.optim.lr_scheduler` 做学习率衰减。
5. **正则化**：在隐藏层后加 `nn.Dropout(p=0.2)` 缓解过拟合；或使用 weight decay。
6. **批大小**：对比 batch_size = 32 / 128 / 512 的训练速度和效果。
7. **训练轮数**：观察更多 epoch 下的训练/测试准确率变化。